# M1M3 True-component Hybrid EnbPI benchmark

這是 KF decomposition 的 paired ablation。Current 組使用 causal KF estimated low/high；True-component 組跳過 KF，改用模擬的 true low/high 歷史，但兩組使用完全相同的 ARIMA、ANN、moving-block bootstrap、nested OOB、bias correction 與 EnbPI。比較表另外列出 Equation Oracle RMSE，形成三層比較。

In [ ]:
import importlib
import pandas as pd
from kf_forecasting.models import kf_enbpi, kf_true_component_benchmark
importlib.reload(kf_enbpi)
importlib.reload(kf_true_component_benchmark)

from kf_forecasting.models.kf_enbpi import EnbPIConfig
from kf_forecasting.models.kf_true_component_benchmark import (
    run_paired_true_component_experiment, paired_comparison_table,
    paired_true_component_monte_carlo, plot_paired_comparison,
    plot_representative_pair,
)
pd.set_option('display.precision', 4)
print('Loaded:', kf_enbpi.__file__)
print('Loaded:', kf_true_component_benchmark.__file__)

## Paired configuration
兩組共用相同 data seed、ensemble seed 與全部 predictor/EnbPI 設定，唯一差異是 component 來源。

In [ ]:
config = EnbPIConfig(
    window_size=15, alpha=0.05, n_bootstrap=30, block_length=None,
    batch_size=1, beta_grid_size=101,
    oob_bias_correction=True, oob_bias_correction_mode='combined',
    arima_order=None, arima_max_p=4, arima_max_q=4,
    ann_hidden_layers=(32, 16), ann_max_iter=500, ann_alpha=1e-4,
    ann_learning_rate_init=1e-3, ann_target_standardization=True,
    ann_early_stopping=False, ann_rolling_validation=True,
    ann_rolling_splits=3, ann_validation_fraction=0.10,
    ann_iteration_candidates=(125, 250, 500), ann_tol=1e-3,
    random_state=1234,
)
train_size = 650
horizon = 50
n_runs = 20

## Single paired run

In [ ]:
current_result, true_component_result = run_paired_true_component_experiment(
    'm1m3', train_size=train_size, horizon=horizon,
    config=config, data_seed=2026,
)
display(paired_comparison_table(current_result, true_component_result))
plot_paired_comparison(current_result, true_component_result);

## Paired Monte Carlo
正的 `rmse_gain_kf_minus_true_component` 代表 true-component Hybrid 比 Current KF Hybrid 更準。

In [ ]:
paired_runs, paired_summary, current_results, true_component_results = (
    paired_true_component_monte_carlo(
        'm1m3', n_runs=n_runs, train_size=train_size, horizon=horizon,
        config=config, seed=2026,
    )
)
display(paired_summary)
display(paired_runs)
plot_representative_pair(
    paired_runs, current_results, true_component_results, model_name='m1m3'
);